# Loan prediction exploratory data analysis

This notebook describes the **labeled Kaggle training CSV only**. It does not load the Kaggle `test.csv`, create a holdout split, fit preprocessing, choose features, or train models. The source CSV is read without modification; missing values are retained. The notebook's copy used to label missing categories is only for plotting.

All statistics and charts are calculated from the observed CSV. Figures are saved in `reports/figures/`, and a measured summary is written to `reports/eda_report.md`. Statistical outlier flags are descriptive and are not treated as invalid applicants.

In [1]:
from pathlib import Path
import sys
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

def find_project_root(start: Path) -> Path:
    """Find the repository root without relying on the notebook cwd."""
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "config.yaml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find project root containing config.yaml and data/")

PROJECT_ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from config.paths import get_project_paths
from config.settings import load_config
from data.loader import load_csv

CONFIG = load_config()
PROJECT_PATHS = get_project_paths(CONFIG)
LABELED_PATH = PROJECT_PATHS.labeled_dataset
SOURCE_URL = CONFIG["dataset"]["source_url"]
TARGET = CONFIG["dataset"]["target_column"]
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
REPORT_PATH = PROJECT_ROOT / "reports" / "eda_report.md"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if not LABELED_PATH.is_file():
    raise FileNotFoundError(f"Labeled training CSV not found: {LABELED_PATH}")

sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 160, "axes.titleweight": "bold"})
TARGET_ORDER = ["N", "Y"]
TARGET_PALETTE = {"N": "#C44E52", "Y": "#55A868"}
CATEGORY_MISSING = "(Missing)"

# Intentionally load only labeled train data. The separate Kaggle test CSV is never opened here.
train_df = load_csv(LABELED_PATH)
if TARGET not in train_df.columns:
    raise ValueError(f"Required target column {TARGET!r} is missing")
if not set(train_df[TARGET].dropna().unique()).issubset({"Y", "N"}):
    raise ValueError("Loan_Status contains labels outside the observed Y/N contract")

plot_df = train_df.copy()
categorical_columns = [
    "Gender", "Married", "Dependents", "Education", "Self_Employed",
    "Credit_History", "Property_Area", TARGET,
]
for column in categorical_columns:
    if column in plot_df.columns:
        plot_df[column] = plot_df[column].astype("string").fillna(CATEGORY_MISSING)

print(f"Project root: {PROJECT_ROOT}")
print(f"Labeled data only: {LABELED_PATH.name}; shape={train_df.shape}")
print("No holdout or unlabeled Kaggle test data was loaded.")

Project root: C:\Projects\loan_predection_system
Labeled data only: train_u6lujuX_CVtuZ9i.csv; shape=(614, 13)
No holdout or unlabeled Kaggle test data was loaded.


## Dataset schema and numerical statistics

The tables below are raw descriptive summaries. Pandas excludes missing numeric values from numeric statistics; missingness is reported separately. No values are imputed, clipped, or filtered.

In [2]:
print(f"Rows: {len(train_df):,}; columns: {len(train_df.columns)}")
schema = pd.DataFrame({
    "dtype": train_df.dtypes.astype(str),
    "non_missing": train_df.notna().sum(),
    "missing": train_df.isna().sum(),
    "missing_percent": (100 * train_df.isna().mean()).round(2),
    "unique_non_missing": train_df.nunique(dropna=True),
})
display(schema)

numeric_columns = train_df.select_dtypes(include="number").columns.tolist()
numeric_stats = train_df[numeric_columns].describe().T
display(numeric_stats.round(3))

category_columns = [column for column in categorical_columns if column != TARGET]
category_distributions = {
    column: train_df[column].value_counts(dropna=False).rename_axis("value").to_frame("count")
    for column in category_columns
}
for column, distribution in category_distributions.items():
    print(f"\n{column} distribution (missing shown explicitly):")
    display(distribution)

Rows: 614; columns: 13


,dtype,non_missing,missing,missing_percent,unique_non_missing
Loan_ID,str,614,0,0.00,614
Gender,str,601,13,2.12,2
Married,str,611,3,0.49,2
Dependents,str,599,15,2.44,4
Education,str,614,0,0.00,2
Self_Employed,str,582,32,5.21,2
ApplicantIncome,int64,614,0,0.00,505
CoapplicantIncome,float64,614,0,0.00,287
LoanAmount,float64,592,22,3.58,203
Loan_Amount_Term,float64,600,14,2.28,10


,count,mean,std,min,25%,50%,75%,max
ApplicantIncome,614.0,5403.459,6109.042,150.0,2877.5,3812.5,5795.00,81000.0
CoapplicantIncome,614.0,1621.246,2926.248,0.0,0.0,1188.5,2297.25,41667.0
LoanAmount,592.0,146.412,85.587,9.0,100.0,128.0,168.00,700.0
Loan_Amount_Term,600.0,342.000,65.120,12.0,360.0,360.0,360.00,480.0
Credit_History,564.0,0.842,0.365,0.0,1.0,1.0,1.00,1.0



Gender distribution (missing shown explicitly):


,count
value,
Male,489
Female,112
NaN,13



Married distribution (missing shown explicitly):


... (7 outputs left out)


Credit_History distribution (missing shown explicitly):


,count
value,
1.0,475
0.0,89
NaN,50



Property_Area distribution (missing shown explicitly):


,count
value,
Semiurban,233
Urban,202
Rural,179


## Approval, income, loan, credit, employment, and property patterns

Bars and distributions below use observed training rows. Outcome-rate charts include missing category values as their own group. Numeric histograms retain the full observed scale; skew and extreme values are not removed.

In [3]:
def save_figure(fig: plt.Figure, filename: str) -> Path:
    """Save a tight-layout chart beneath the project figure directory."""
    output = FIGURES_DIR / filename
    fig.savefig(output, bbox_inches="tight")
    plt.close(fig)
    return output

def approval_rate_table(column: str) -> pd.DataFrame:
    """Compute observed approval rate by category, retaining missing values."""
    groups = plot_df.groupby(column, dropna=False, observed=False)[TARGET]
    rates = groups.apply(lambda values: (values == "Y").mean()).rename("Approval rate")
    counts = groups.size().rename("Applications")
    return pd.concat([rates, counts], axis=1).reset_index()

def plot_approval_rates(ax: plt.Axes, column: str, title: str) -> None:
    """Plot actual approval share for each observed category."""
    rates = approval_rate_table(column)
    sns.barplot(data=rates, x=column, y="Approval rate", color="#4C72B0", ax=ax)
    ax.set_title(title)
    ax.set_xlabel(column.replace("_", " "))
    ax.set_ylabel("Approved applications / applications")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=20)
    ax.legend(
        handles=[Patch(color="#4C72B0", label="Observed approval rate (Y)")],
        title="Outcome",
        loc="best",
    )
    for bar, count in zip(ax.patches, rates["Applications"]):
        ax.annotate(
            f"n={int(count)}",
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8,
        )

# 1. Loan approval versus rejection.
fig, ax = plt.subplots(figsize=(7, 5))
sns.countplot(
    data=plot_df, x=TARGET, hue=TARGET, order=TARGET_ORDER,
    hue_order=TARGET_ORDER, palette=TARGET_PALETTE, dodge=False, ax=ax,
)
ax.set_title("Loan approval versus rejection (labeled training data)")
ax.set_xlabel("Loan status")
ax.set_ylabel("Applications")
ax.set_xticks([0, 1], ["Rejected (N)", "Approved (Y)"])
ax.legend(title="Loan status", labels=["Rejected (N)", "Approved (Y)"])
for bar in ax.patches:
    ax.annotate(
        f"{int(bar.get_height())}",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 3), textcoords="offset points", ha="center",
    )
save_figure(fig, "01_approval_vs_rejection.png")

# 2. Approval patterns by credit history, with missing history explicit.
fig, ax = plt.subplots(figsize=(8, 5))
plot_approval_rates(ax, "Credit_History", "Approval rate by credit history")
save_figure(fig, "02_approval_by_credit_history.png")

# 3. Full, unfiltered income and loan amount/term distributions.
distribution_columns = [
    ("ApplicantIncome", "Applicant income"),
    ("CoapplicantIncome", "Co-applicant income"),
    ("LoanAmount", "Loan amount"),
    ("Loan_Amount_Term", "Loan term (months)"),
]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (column, label) in zip(axes.flat, distribution_columns):
    sns.histplot(
        data=train_df, x=column, hue=TARGET, hue_order=TARGET_ORDER,
        palette=TARGET_PALETTE, bins=30, element="step", stat="count",
        common_norm=False, ax=ax,
    )
    ax.set_title(f"{label} distribution by observed loan status")
    ax.set_xlabel(label)
    ax.set_ylabel("Applications")
    if column == "Loan_Amount_Term":
        ax.set_xticks(sorted(train_df[column].dropna().unique()))
        ax.tick_params(axis="x", rotation=35)
    if ax.get_legend() is not None:
        ax.get_legend().set_title("Loan status")
fig.suptitle("Income, loan amount, and loan term distributions", fontsize=16)
fig.tight_layout()
save_figure(fig, "03_income_loan_distributions.png")

# 4. Approval patterns by employment status, including missing status.
fig, ax = plt.subplots(figsize=(8, 5))
plot_approval_rates(ax, "Self_Employed", "Approval rate by self-employment status")
save_figure(fig, "04_approval_by_employment.png")

# 5. Correlations of numeric fields only; no holdout data and no feature selection.
correlation = train_df[numeric_columns].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    correlation, annot=True, fmt=".2f", cmap="vlag", center=0,
    vmin=-1, vmax=1, square=True,
    cbar_kws={"label": "Pearson correlation"}, ax=ax,
)
ax.set_title("Numerical feature correlation (labeled training data)")
ax.set_xlabel("Numeric field")
ax.set_ylabel("Numeric field")
save_figure(fig, "05_numerical_correlation.png")

# 6. Approval patterns by property area.
fig, ax = plt.subplots(figsize=(8, 5))
plot_approval_rates(ax, "Property_Area", "Approval rate by property area")
save_figure(fig, "06_approval_by_property_area.png")

# Additional descriptive plots: category counts (missing shown) and missingness.
category_plot_columns = [
    "Gender", "Married", "Dependents", "Education",
    "Self_Employed", "Credit_History", "Property_Area", TARGET,
]
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, column in zip(axes.flat, category_plot_columns):
    counts = plot_df[column].value_counts(dropna=False)
    categories = counts.index.astype(str).tolist()
    colors = ["#999999" if value == CATEGORY_MISSING else "#4C72B0" for value in categories]
    ax.bar(categories, counts.values, color=colors, label="Observed records")
    ax.set_title(f"{column.replace('_', ' ')} distribution")
    ax.set_xlabel("Category (missing kept separate)")
    ax.set_ylabel("Applications")
    ax.tick_params(axis="x", rotation=30)
    ax.legend(handles=[Patch(color="#4C72B0", label="Observed"), Patch(color="#999999", label="Missing")], title="Value")
fig.suptitle("Categorical distributions in labeled training data", fontsize=16)
fig.tight_layout()
save_figure(fig, "07_categorical_distributions.png")

missing_counts = train_df.isna().sum().sort_values(ascending=False)
missing_counts = missing_counts[missing_counts > 0]
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(missing_counts.index, missing_counts.values, color="#DD8452", label="Missing values")
ax.set_title("Missing values by field (labeled training data)")
ax.set_xlabel("Field")
ax.set_ylabel("Missing observations")
ax.tick_params(axis="x", rotation=35)
ax.legend(title="Observation status")
save_figure(fig, "08_missing_values.png")

print(f"Saved {len(list(FIGURES_DIR.glob('*.png')))} figures to {FIGURES_DIR}")

Saved 8 figures to C:\Projects\loan_predection_system\reports\figures


## Persist the measured EDA report

The report below is assembled from the notebook's in-memory labeled dataframe and descriptive tables. It documents the scope and limitations of these univariate and group summaries.

In [4]:
def markdown_table(headers: list[str], rows: list[list[object]]) -> str:
    """Render simple Markdown tables without an optional tabulate dependency."""
    lines = ["| " + " | ".join(headers) + " |", "|" + "|".join(["---"] * len(headers)) + "|"]
    lines.extend("| " + " | ".join(str(value) for value in row) + " |" for row in rows)
    return "\n".join(lines)

numeric_report = numeric_stats.copy().round(3).reset_index(names="field")
numeric_table = markdown_table(
    numeric_report.columns.astype(str).tolist(),
    numeric_report.astype(object).where(pd.notna(numeric_report), "—").values.tolist(),
)
target_counts = train_df[TARGET].value_counts().reindex(TARGET_ORDER, fill_value=0)
target_rows = [
    ["N (rejected)", int(target_counts["N"]), f"{target_counts['N'] / len(train_df):.2%}"],
    ["Y (approved)", int(target_counts["Y"]), f"{target_counts['Y'] / len(train_df):.2%}"],
]
target_table = markdown_table(["Loan status", "Applications", "Share"], target_rows)

missing_rows = [
    [column, int(train_df[column].isna().sum()), f"{train_df[column].isna().mean():.2%}"]
    for column in train_df.columns if train_df[column].isna().any()
]
missing_table = markdown_table(["Field", "Missing", "Missing share"], missing_rows)

categorical_rows = []
for column in category_columns:
    values = train_df[column].value_counts(dropna=False)
    for value, count in values.items():
        label = CATEGORY_MISSING if pd.isna(value) else str(value)
        categorical_rows.append([column, label, int(count), f"{count / len(train_df):.2%}"])
categorical_table = markdown_table(
    ["Field", "Observed category", "Applications", "Share of rows"], categorical_rows
)

figure_files = [
    ("01_approval_vs_rejection.png", "Loan approval versus rejection"),
    ("02_approval_by_credit_history.png", "Approval rate by credit history"),
    ("03_income_loan_distributions.png", "Income, loan amount, and term distributions"),
    ("04_approval_by_employment.png", "Approval rate by employment status"),
    ("05_numerical_correlation.png", "Numerical correlation heatmap"),
    ("06_approval_by_property_area.png", "Approval rate by property area"),
    ("07_categorical_distributions.png", "Categorical distributions"),
    ("08_missing_values.png", "Missing value counts"),
]
figure_list = "\n".join(
    f"- [{title}](figures/{filename})" for filename, title in figure_files
)
report_text = f"""# Exploratory data analysis

**Dataset:** [Kaggle Loan Prediction Problem Dataset]({SOURCE_URL})  
**File analyzed:** `{LABELED_PATH.relative_to(PROJECT_ROOT)}` (labeled training CSV only)  
**Scope:** {len(train_df):,} rows, {len(train_df.columns)} columns. The separate Kaggle test CSV and any holdout partition were not loaded or analyzed. No preprocessing or model selection was performed.

## Target distribution

{target_table}

## Numerical descriptive statistics

Statistics use the observed values; missing entries are excluded per field. Income and loan values are strongly right-skewed in this sample, so means and maxima should be read alongside medians and quartiles. Distribution plots retain all observed values.

{numeric_table}

## Categorical distributions

Missing values are represented as `{CATEGORY_MISSING}` in this table and in categorical plots; the source frame is unchanged.

{categorical_table}

## Missing value patterns

{missing_table}

Credit history is observed for {train_df['Credit_History'].notna().sum():,} of {len(train_df):,} labeled applications. Employment status (`Self_Employed`) is missing for {train_df['Self_Employed'].isna().sum()} rows. The missingness plot reports observed nulls; no missing values were filled.

## Approval patterns

Approval-rate figures show the observed share with `Loan_Status = Y` by category, including missing categories where applicable. These are descriptive associations, not causal effects. The correlation heatmap uses numeric columns only and pairwise available values; correlations are not predictive performance or causal evidence.

## Figures

{figure_list}

## Interpretation and limitations

- EDA used only the labeled Kaggle training CSV. The Kaggle test CSV and any future holdout set were excluded.
- No imputation, filtering, outlier deletion, resampling, or source modification was performed.
- Long-tailed income and loan observations are shown as observed. Extreme values may be unusual but valid applicants; visual/statistical extremes alone do not establish data errors.
- Category-level approval differences are descriptive and may reflect confounding, missingness, or sample size. They are not lending rules.
- The dataset is small and contains missing values; findings may not generalize to other populations or current lending conditions.
"""

REPORT_PATH.write_text(report_text, encoding="utf-8")
print(f"Wrote EDA report: {REPORT_PATH}")
print(f"Wrote figures: {FIGURES_DIR}")
print(f"Approval counts: {target_counts.to_dict()}")
display(numeric_stats.round(3))

Wrote EDA report: C:\Projects\loan_predection_system\reports\eda_report.md
Wrote figures: C:\Projects\loan_predection_system\reports\figures
Approval counts: {'N': 192, 'Y': 422}


,count,mean,std,min,25%,50%,75%,max
ApplicantIncome,614.0,5403.459,6109.042,150.0,2877.5,3812.5,5795.00,81000.0
CoapplicantIncome,614.0,1621.246,2926.248,0.0,0.0,1188.5,2297.25,41667.0
LoanAmount,592.0,146.412,85.587,9.0,100.0,128.0,168.00,700.0
Loan_Amount_Term,600.0,342.000,65.120,12.0,360.0,360.0,360.00,480.0
Credit_History,564.0,0.842,0.365,0.0,1.0,1.0,1.00,1.0
